In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import torch
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import glob
from PIL import Image

In [ ]:
P3M_ROOT = "/content/drive/MyDrive/DATA/P3M-10k" 

RVM_WEIGHT_PATH = "/content/drive/MyDrive/RVM/rvm_resnet50.pth"

GENERATE_SEM_LABEL = False     
GENERATE_BASE_ALPHA = True    

TEST_MODE = True         
TEST_SIZE = 9421              
RGB_SAVE_LIMIT = 1          

ALPHA_BG_MAX = 0.05            
ALPHA_TR_BACK_MAX = 0.5        
ALPHA_TR_FORE_MIN = 0.5       
ALPHA_FORE_MIN = 0.95          

COLOR_MAP = {
    0: [0, 0, 0],          
    1: [255, 0, 0],        
    2: [0, 0, 255],         
    3: [255, 255, 255]      
}

print(f"Path: {P3M_ROOT}")
print(f"RVM Weight: {RVM_WEIGHT_PATH}")
print(f"Generate sem label: {GENERATE_SEM_LABEL}")
print(f"generate base alpha: {GENERATE_BASE_ALPHA}")
print(f"test mode: {TEST_MODE} ({TEST_SIZE})")
print(f"RGB save {RGB_SAVE_LIMIT} ")

In [ ]:
def load_rvm_model(weight_path):
    try:
        model = torch.hub.load("PeterL1n/RobustVideoMatting", "resnet50") 

        if os.path.exists(weight_path):
            weights = torch.load(weight_path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
            model.load_state_dict(weights)
            print(f"RVM success: {weight_path}")
        else:
            print(f"No exist: {weight_path}, use pretrained weight")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)
        model.eval()
        print(f"{device}")

        return model, device
    except Exception as e:
        print(f"{e}")
        return None, None

In [ ]:
def alpha_to_4class_semantic(alpha_path, save_rgb_path=None, save_label_path=None):
    try:
        alpha = cv2.imread(alpha_path, cv2.IMREAD_GRAYSCALE)
        if alpha is None:
            return False, None

        if alpha.max() > 1:
            alpha = alpha.astype(np.float32) / 255.0

        h, w = alpha.shape
        semantic_label = np.zeros((h, w), dtype=np.uint8)

        mask_tr_back = (alpha > ALPHA_BG_MAX) & (alpha <= ALPHA_TR_BACK_MAX)
        semantic_label[mask_tr_back] = 1

        mask_tr_fore = (alpha > ALPHA_TR_FORE_MIN) & (alpha < ALPHA_FORE_MIN)
        semantic_label[mask_tr_fore] = 2

        mask_fore = (alpha >= ALPHA_FORE_MIN)
        semantic_label[mask_fore] = 3

        if save_rgb_path:
            rgb_output = np.zeros((h, w, 3), dtype=np.uint8)
            for label, color in COLOR_MAP.items():
                rgb_output[semantic_label == label] = color
            os.makedirs(os.path.dirname(save_rgb_path), exist_ok=True)
            cv2.imwrite(save_rgb_path, rgb_output)

        if save_label_path:
            os.makedirs(os.path.dirname(save_label_path), exist_ok=True)
            cv2.imwrite(save_label_path, semantic_label)

        return True, semantic_label

    except Exception as e:
        print(f"{e}")
        return False, None

In [ ]:
def predict_alpha_with_rvm(model, device, img_path, save_alpha_path=None):
    try:
        img = cv2.imread(img_path)
        if img is None:
            return False, None

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        src = torch.from_numpy(img_rgb).float() / 255.0
        src = src.permute(2, 0, 1).unsqueeze(0)
        src_bgr = src[:, [2, 1, 0], :, :]
        src_bgr = src_bgr.to(device)

        with torch.no_grad():
            fgr, pha = model(src_bgr, None)[:2] 
            alpha = pha[0, 0].cpu().numpy()

        alpha = np.clip(alpha, 0, 1)

        if save_alpha_path:
            os.makedirs(os.path.dirname(save_alpha_path), exist_ok=True)
            alpha_uint8 = (alpha * 255).astype(np.uint8)
            cv2.imwrite(save_alpha_path, alpha_uint8)

        return True, alpha

    except Exception as e:
        print(f"{e}")
        return False, None

In [ ]:
def process_single_file(args):
    (alpha_path, img_path, sem_rgb_path, sem_label_path, base_alpha_path, idx,
     generate_sem, generate_base, model, device) = args

    results = {"sem_label": False, "base_alpha": False}

    if generate_sem and alpha_path and os.path.exists(alpha_path):
        rgb_path = sem_rgb_path if (idx < RGB_SAVE_LIMIT) else None
        success, _ = alpha_to_4class_semantic(alpha_path, rgb_path, sem_label_path)
        results["sem_label"] = success

    if generate_base and img_path and os.path.exists(img_path) and model is not None:
        success, _ = predict_alpha_with_rvm(model, device, img_path, base_alpha_path)
        results["base_alpha"] = success

    return alpha_path, results

In [ ]:
def process_dataset(mask_dir, img_dir, sem_rgb_dir, sem_label_dir, base_alpha_dir,
                    model, device, dataset_name, max_workers=8):
    mask_paths = []
    img_paths = []

    for ext in ['*.png', '*.jpg', '*.jpeg']:
        mask_paths.extend(glob.glob(os.path.join(mask_dir, ext)))

    if img_dir and os.path.exists(img_dir):
        for ext in ['*.png', '*.jpg', '*.jpeg']:
            img_paths.extend(glob.glob(os.path.join(img_dir, ext)))

    mask_dict = {os.path.basename(p).split('.')[0]: p for p in mask_paths}
    img_dict = {os.path.basename(p).split('.')[0]: p for p in img_paths}

    all_keys = set(mask_dict.keys())

    if not all_keys:
        print(f"Not found: {mask_dir}")
        return 0, 0, 0, 0

    all_keys = sorted(list(all_keys))

    if TEST_MODE:
        all_keys = all_keys[:TEST_SIZE]
        print(f"Find {len(mask_dict)} GT alpha, test mode {len(all_keys)}")
    else:
        print(f"Find {len(mask_dict)} GT alpha, all")

    tasks = []
    for idx, key in enumerate(all_keys):
        alpha_path = mask_dict.get(key, None)
        img_path = img_dict.get(key, None)

        sem_rgb_path = os.path.join(sem_rgb_dir, f"{key}.png") if GENERATE_SEM_LABEL else None
        sem_label_path = os.path.join(sem_label_dir, f"{key}.png") if GENERATE_SEM_LABEL else None
        base_alpha_path = os.path.join(base_alpha_dir, f"{key}.png") if GENERATE_BASE_ALPHA else None

        tasks.append((alpha_path, img_path, sem_rgb_path, sem_label_path, base_alpha_path, idx,
                      GENERATE_SEM_LABEL, GENERATE_BASE_ALPHA, model, device))

    sem_success = 0
    base_success = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_single_file, task): task for task in tasks}

        with tqdm(total=len(tasks), desc=f" {dataset_name}") as pbar:
            for future in as_completed(futures):
                _, results = future.result()
                if results["sem_label"]:
                    sem_success += 1
                if results["base_alpha"]:
                    base_success += 1
                pbar.update(1)

    print(f"sem label: {sem_success}/{len(tasks)}")
    print(f"base_alpha (RVM): {base_success}/{len(tasks)}")

    return len(tasks), sem_success, base_success

In [ ]:
def main():
    global GENERATE_BASE_ALPHA
    print("=" * 60)
    print(" P3M-10k data preprocess")
    print(f"   sem_label: {GENERATE_SEM_LABEL}")
    print(f"   base_alpha: {GENERATE_BASE_ALPHA}")
    print("=" * 60)

    model = None
    device = None
    if GENERATE_BASE_ALPHA:
        model, device = load_rvm_model(RVM_WEIGHT_PATH)
        if model is None:
            print("Fail")
            GENERATE_BASE_ALPHA = False

    datasets = []

    train_mask = os.path.join(P3M_ROOT, "train", "mask")
    train_img = os.path.join(P3M_ROOT, "train", "blurred_image")
    train_sem_rgb = os.path.join(P3M_ROOT, "train", "sem_rgb")
    train_sem_label = os.path.join(P3M_ROOT, "train", "sem_label")
    train_base_alpha = os.path.join(P3M_ROOT, "train", "base_alpha")

    if os.path.exists(train_mask):
        datasets.append(("Train", train_mask, train_img, train_sem_rgb, train_sem_label, train_base_alpha))
        print(f"Add Train")

    val_p_mask = os.path.join(P3M_ROOT, "validation", "P3M-500-P", "mask")
    val_p_img = os.path.join(P3M_ROOT, "validation", "P3M-500-P", "blurred_image")
    val_p_sem_rgb = os.path.join(P3M_ROOT, "validation", "P3M-500-P", "sem_rgb")
    val_p_sem_label = os.path.join(P3M_ROOT, "validation", "P3M-500-P", "sem_label")
    val_p_base_alpha = os.path.join(P3M_ROOT, "validation", "P3M-500-P", "base_alpha")

    if os.path.exists(val_p_mask):
        datasets.append(("Val-P3M-500-P", val_p_mask, val_p_img, val_p_sem_rgb, val_p_sem_label, val_p_base_alpha))
        print(f"Add Validation P3M-500-P")

    t_p_mask = os.path.join(P3M_ROOT, "validation", "P3M-500-NP", "mask")
    t_p_img = os.path.join(P3M_ROOT, "validation", "P3M-500-NP", "original_image")
    t_p_sem_rgb = os.path.join(P3M_ROOT, "validation", "P3M-500-NP", "sem_rgb")
    t_p_sem_label = os.path.join(P3M_ROOT, "validation", "P3M-500-NP", "sem_label")
    t_p_base_alpha = os.path.join(P3M_ROOT, "validation", "P3M-500-NP", "base_alpha")

    if os.path.exists(val_p_mask):
        datasets.append(("Val-P3M-500-NP", t_p_mask, t_p_img, t_p_sem_rgb, t_p_sem_label, t_p_base_alpha))
        print(f"Add Test P3M-500-NP")


    if not datasets:
        print("\nNot found")
        return

    total_files = 0
    total_sem = 0
    total_base = 0

    for name, mask_dir, img_dir, sem_rgb_dir, sem_label_dir, base_alpha_dir in datasets:
        print(f"\n📁 {name}")
        print(f"   Mask: {mask_dir}")
        print(f"   Origin: {img_dir}")

        if GENERATE_SEM_LABEL:
            print(f"   Sem RGB out: {sem_rgb_dir} ({RGB_SAVE_LIMIT})")
            print(f"   Sem label out: {sem_label_dir}")
        if GENERATE_BASE_ALPHA:
            print(f"   base_alpha out: {base_alpha_dir}")

        n_files, n_sem, n_base = process_dataset(
            mask_dir, img_dir, sem_rgb_dir, sem_label_dir, base_alpha_dir,
            model, device, name, max_workers=4  
        )

        total_files += n_files
        total_sem += n_sem
        total_base += n_base
        
    print("\n" + "=" * 60)
    print("Complete")
    print(f"   # of files: {total_files}")
    if GENERATE_SEM_LABEL:
        print(f"   sem label success: {total_sem}")
    if GENERATE_BASE_ALPHA:
        print(f"   base_alpha success: {total_base}")
    print("=" * 60)

In [ ]:
if __name__ == "__main__":
    main()